In [1]:
import warnings
warnings.simplefilter(action='ignore')

import os
import requests
import joblib

import numpy
import pandas

import matplotlib.pyplot as plt
import scipy.stats as stats
import tensorflow_decision_forests as tfdf

In [2]:
from catboost import CatBoostRegressor 
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.svm import SVR

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.ensemble import AdaBoostRegressor

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import accuracy_score

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import SelectFromModel

from sklearn.impute import SimpleImputer
import sklearn.linear_model as linear_model
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [3]:
train = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/train.csv').drop(columns=['ID','efs_time'])
test = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv').drop(columns=['ID'])

In [4]:
encoder = LabelEncoder()
normalize = SimpleImputer(strategy='mean')

In [5]:
for i in zip(train.columns,train.dtypes):
    if i[1]=='O':
        train[i[0]]=normalize.fit_transform(encoder.fit_transform(train[i[0]].to_numpy().reshape(-1,1)).reshape(-1,1))
    else:
        train[i[0]].fillna(train[i[0]].mean(), inplace=True)
        train[i[0]]=normalize.fit_transform(numpy.array(train[i[0]]).reshape(-1,1))
for i in zip(test.columns,test.dtypes):
    if i[1]=='O':
        test[i[0]]=normalize.fit_transform(numpy.array(encoder.fit_transform(test[i[0]].to_numpy().reshape(-1,1))).reshape(-1,1))
    else:
        test[i[0]].fillna(test[i[0]].mean(), inplace=True)
        test[i[0]]=normalize.fit_transform(numpy.array(test[i[0]]).reshape(-1,1))

In [6]:
train = train.fillna(0)
test = test.fillna(0)

train_x = train.drop(columns=['efs'])
train_y = train['efs']

for i in test:
    test[i]=test[i].to_numpy()
    
#train.hist(figsize=(16, 20), bins=50, xlabelsize=8, ylabelsize=8)

clf = GradientBoostingRegressor(n_estimators=100).fit(train_x, train_y)
model = SelectFromModel(clf, prefit=True)

weight = model.transform(train_x).reshape(-1)
len(train.columns)

58

In [7]:
Fold=5
kf = KFold(n_splits=Fold)

LGBM_pre_train = numpy.zeros(len(train))
LGBM_pre_test = numpy.zeros(len(test))

xgb_pre_train = numpy.zeros(len(train))
xgb_pre_test = numpy.zeros(len(test))

cat_pre_train = numpy.zeros(len(train))
cat_pre_test = numpy.zeros(len(test))

RandomF_pre_train = numpy.zeros(len(train))
RandomF_pre_test = numpy.zeros(len(test))

for i, (train_index, test_index) in enumerate(kf.split(train)):

    print(f"FOLD: {i}")

    x_fold=train_x[train_index[0]: train_index[len(train_index)-1]]
    y_fold=train_y[train_index[0]: train_index[len(train_index)-1]]

    x_test_fold=train_x[test_index[0]: test_index[len(test_index)-1]]
    y_test_fold=train_y[test_index[0]: test_index[len(test_index)-1]]

    weight_ = weight[train_index[0]: train_index[len(train_index)-1]]

    lightgbm = LGBMRegressor()
    xgboost = XGBRegressor()
    catboost = CatBoostRegressor(learning_rate=0.009,
                                 depth=5,
                                 l2_leaf_reg=3.5,
                                 min_child_samples=32,
                                 grow_policy='Depthwise',
                                 iterations=1000,
                                 eval_metric='RMSE',
                                 od_type='Iter',
                                 od_wait=50,
                                 random_state=42,
                                 logging_level='Silent')
    RandomF = GradientBoostingRegressor()
    
    lightgbm.fit(x_fold, y_fold, eval_sample_weight=weight_)
    xgboost.fit(x_fold, y_fold, sample_weight=weight_)
    catboost.fit(x_fold, y_fold, sample_weight=weight_)
    RandomF.fit(x_fold, y_fold, weight_)

    LGBM_pre_train+=lightgbm.predict(train_x)
    LGBM_pre_test+=lightgbm.predict(test)

    xgb_pre_train+=xgboost.predict(train_x)
    xgb_pre_test+=xgboost.predict(test)

    cat_pre_train+=catboost.predict(train_x)
    cat_pre_test+=catboost.predict(test)

    RandomF_pre_train += RandomF.predict(train_x)
    RandomF_pre_test += RandomF.predict(test)

    print(RandomF.score(x_test_fold, y_test_fold))
    print(catboost.score(x_test_fold, y_test_fold))
    print(xgboost.score(x_test_fold, y_test_fold))
    print(lightgbm.score(x_test_fold, y_test_fold))

LGBM_pre_train = LGBM_pre_train/Fold
LGBM_pre_test = LGBM_pre_test/Fold

FOLD: 0
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009086 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 830
[LightGBM] [Info] Number of data points in the train set: 23039, number of used features: 57
[LightGBM] [Info] Start training from score 0.539780
0.15503938549527596
0.14167465660987755
0.0016986869042473485
0.1897775971941208
FOLD: 1
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012865 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 830
[LightGBM] [Info] Number of data points in the train set: 28799, number of used features: 57
[LightGBM] [Info] Start training from score 0.539324
0.16157895400510902
0.13434934755662697
0.1939547630225651
0.2917709292214222
FOLD: 2
[LightG

In [8]:
new_train_data = pandas.DataFrame({'RandomF' : RandomF_pre_train,
                             'LGBM' : LGBM_pre_train,
                             'xgb' : xgb_pre_train,
                             'cat' : cat_pre_train})

new_test_data = pandas.DataFrame({'RandomF' : RandomF_pre_test,
                             'LGBM' : LGBM_pre_test,
                             'xgb' : xgb_pre_test,
                             'cat' : cat_pre_test})
model=GradientBoostingRegressor()
model.fit(new_train_data, train_y)

GradientBoostingRegressor()

In [9]:
id = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv')['ID']
test_predictions = ((RandomF_pre_test/Fold)+(LGBM_pre_test/Fold)+(xgb_pre_test/Fold)+(cat_pre_test/Fold))/4
#((RandomF_pre_test/Fold)+(LGBM_pre_test/Fold)+(xgb_pre_test/Fold)+(cat_pre_test/Fold))/4
#model.predict(new_test_data)
#((RandomF_pre_test/10)+(LGBM_pre_test/5)+(xgb_pre_test/5)+(cat_pre_test/5))/4
test_predictions

array([0.16222302, 0.41182892, 0.03005511])

In [10]:
submission = pandas.DataFrame({
    'ID': id.values,
    'prediction': test_predictions,
})
submission

,ID,prediction
0,28800,0.162223
1,28801,0.411829
2,28802,0.030055


In [11]:
submission.to_csv('submission.csv', index = False)